# Программная генерация атак и стресс-тестирование классификатора ResNet-18

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json
import random
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from IPython.display import display

plt.rcParams["figure.figsize"] = (10, 5)

@dataclass
class AttackConfig:
    ROOT_DIR: str = "."
    DATA_DIR: str = "runs_cascade/attack_eval"
    DATA_CSV: str = None  # готовый CSV с колонками crop_path и label
    CASCADE_RUNS_DIR: str = "runs_cascade_yolov8s"
    VIDEO_LABELS_CSV: str = ""
    VIDEO_LABELS: dict | None = None  # например {"red_clip_01": "red", "green_clip_02": "green"}
    CASCADE_MATCH_MODE: str = "smooth"
    WEIGHTS_PATH: str = "tl_custom_train/resnet18_tl_custom_state_dict.pt"
    META_PATH: str = "tl_custom_train/resnet18_tl_custom_meta.json"
    OUT_DIR: str = "attack_experiments"
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 0
    SEED: int = 42
    USE_ONLY_BASELINE_CORRECT: bool = True
    MAX_SAMPLES: int | None = None
    VIS_SAMPLES_PER_SETTING: int = 6

    # Параметры атак
    EPSILONS: tuple = (2/255, 4/255, 8/255)
    BIM_ALPHA: float = 1/255
    BIM_STEPS: int = 10
    PGD_ALPHA: float = 1/255
    PGD_STEPS: int = 10
    PGD_RANDOM_START: bool = True

cfg = AttackConfig()
random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

ROOT = Path(cfg.ROOT_DIR)
OUT_DIR = ROOT / cfg.OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print(json.dumps(asdict(cfg), ensure_ascii=False, indent=2))


## 1. Загрузка метаданных и модели

In [ ]:
DEFAULT_CLASS_TO_IDX = {"red": 0, "green": 1}
DEFAULT_IDX_TO_CLASS = {0: "red", 1: "green"}
DEFAULT_INPUT_SIZE = 96
DEFAULT_MEAN = (0.485, 0.456, 0.406)
DEFAULT_STD = (0.229, 0.224, 0.225)

def load_meta(meta_path: Path):
    if meta_path.exists():
        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)
    else:
        meta = {}

    class_to_idx = meta.get("class_to_idx", DEFAULT_CLASS_TO_IDX)
    idx_to_class_raw = meta.get("idx_to_class", DEFAULT_IDX_TO_CLASS)
    idx_to_class = {int(k): v for k, v in idx_to_class_raw.items()} if isinstance(idx_to_class_raw, dict) else DEFAULT_IDX_TO_CLASS

    input_size = int(meta.get("input_size", DEFAULT_INPUT_SIZE))
    mean = tuple(meta.get("imagenet_mean", DEFAULT_MEAN))
    std = tuple(meta.get("imagenet_std", DEFAULT_STD))

    return {
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "input_size": input_size,
        "mean": mean,
        "std": std,
        "raw_meta": meta,
    }

meta_info = load_meta(ROOT / cfg.META_PATH)
CLASS_TO_IDX = meta_info["class_to_idx"]
IDX_TO_CLASS = meta_info["idx_to_class"]
INPUT_SIZE = meta_info["input_size"]
MEAN = meta_info["mean"]
STD = meta_info["std"]

print("CLASS_TO_IDX:", CLASS_TO_IDX)
print("IDX_TO_CLASS:", IDX_TO_CLASS)
print("INPUT_SIZE:", INPUT_SIZE)
print("MEAN:", MEAN)
print("STD:", STD)

val_tf = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def build_model():
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_TO_IDX))
    return model.to(device)

def load_trained_model(weights_path: Path):
    model = build_model()
    state = torch.load(weights_path, map_location=device)
    model.load_state_dict(state)
    model.eval()
    return model

weights_path = ROOT / cfg.WEIGHTS_PATH
assert weights_path.exists(), f"Не найдены веса: {weights_path}"

model = load_trained_model(weights_path)
criterion = nn.CrossEntropyLoss()

print("Модель загружена:", weights_path)

## 2. Формирование набора ROI для атаки


In [ ]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def infer_label_from_path(path: Path):
    tokens = [part.lower() for part in path.parts]
    name = path.name.lower()

    if "red" in tokens or "red" in name:
        return "red"
    if "green" in tokens or "green" in name:
        return "green"
    return None

def normalize_label(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    s = str(value).strip().lower()
    if s in {"red", "green"}:
        return s
    return None

def resolve_path(path_str):
    p = Path(path_str)
    if p.is_absolute():
        return p
    return (ROOT / p).resolve()

def load_video_label_map():
    mapping = {}

    if cfg.VIDEO_LABELS:
        for k, v in cfg.VIDEO_LABELS.items():
            label = normalize_label(v)
            if label is not None:
                mapping[str(k)] = label

    if cfg.VIDEO_LABELS_CSV:
        csv_path = ROOT / cfg.VIDEO_LABELS_CSV
        if csv_path.exists():
            video_df = pd.read_csv(csv_path)
            if not {"video", "label"}.issubset(video_df.columns):
                raise ValueError("В VIDEO_LABELS_CSV нужны колонки video и label")
            for row in video_df.itertuples(index=False):
                label = normalize_label(getattr(row, "label"))
                if label is not None:
                    mapping[str(getattr(row, "video"))] = label

    return mapping

VIDEO_LABEL_MAP = load_video_label_map()

def get_expected_label_for_video(video_name: str, fallback_path: Path | None = None):
    if video_name in VIDEO_LABEL_MAP:
        return VIDEO_LABEL_MAP[video_name]

    stem = Path(video_name).stem
    if stem in VIDEO_LABEL_MAP:
        return VIDEO_LABEL_MAP[stem]

    label = infer_label_from_path(Path(video_name))
    if label is not None:
        return label

    if fallback_path is not None:
        label = infer_label_from_path(fallback_path)
        if label is not None:
            return label

    return None

def collect_from_dir(data_dir: Path):
    rows = []
    for path in sorted(data_dir.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix.lower() not in IMAGE_EXTS:
            continue

        label = None
        parent_name = path.parent.name.lower()
        if parent_name in CLASS_TO_IDX:
            label = parent_name
        else:
            label = infer_label_from_path(path)

        if label is None:
            continue

        rows.append({
            "crop_path": str(path),
            "label": label,
            "source_type": "dir_scan",
            "video": None,
            "frame_idx": None,
            "label_raw": None,
            "label_smooth": None,
        })

    return pd.DataFrame(rows)

def cascade_match_ok(expected_label, label_raw, label_smooth):
    mode = str(cfg.CASCADE_MATCH_MODE).strip().lower()

    raw_ok = normalize_label(label_raw) == expected_label
    smooth_ok = normalize_label(label_smooth) == expected_label

    if mode == "off":
        return True
    if mode == "raw":
        return raw_ok
    if mode == "smooth":
        return smooth_ok
    if mode == "either":
        return raw_ok or smooth_ok
    raise ValueError(f"Неизвестный CASCADE_MATCH_MODE: {cfg.CASCADE_MATCH_MODE}")

def collect_from_cascade_runs(cascade_dir: Path):
    rows = []

    log_files = sorted(cascade_dir.rglob("*_frame_log.csv"))
    if not log_files:
        raise FileNotFoundError(f"В {cascade_dir} не найдено ни одного *_frame_log.csv")

    for log_path in log_files:
        log_df = pd.read_csv(log_path)

        required_cols = {"video", "frame_idx", "crop_path"}
        missing = required_cols - set(log_df.columns)
        if missing:
            raise ValueError(f"В {log_path} нет колонок: {sorted(missing)}")

        for row in log_df.itertuples(index=False):
            crop_path_val = getattr(row, "crop_path", None)
            if crop_path_val is None or (isinstance(crop_path_val, float) and np.isnan(crop_path_val)):
                continue

            crop_path = resolve_path(str(crop_path_val))
            if not crop_path.exists():
                continue
            if crop_path.suffix.lower() not in IMAGE_EXTS:
                continue

            video_name = str(getattr(row, "video"))
            expected_label = get_expected_label_for_video(video_name, fallback_path=crop_path)
            if expected_label is None:
                continue

            label_raw = getattr(row, "label_raw", None)
            label_smooth = getattr(row, "label_smooth", None)

            if not cascade_match_ok(expected_label, label_raw, label_smooth):
                continue

            rows.append({
                "crop_path": str(crop_path),
                "label": expected_label,
                "source_type": "cascade_runs",
                "video": video_name,
                "frame_idx": getattr(row, "frame_idx", None),
                "label_raw": label_raw,
                "label_smooth": label_smooth,
            })

    return pd.DataFrame(rows)

def load_roi_df():
    csv_path = ROOT / cfg.DATA_CSV if cfg.DATA_CSV else None
    cascade_dir = ROOT / cfg.CASCADE_RUNS_DIR if cfg.CASCADE_RUNS_DIR else None
    data_dir = ROOT / cfg.DATA_DIR

    if csv_path and csv_path.exists():
        df = pd.read_csv(csv_path)

        if "crop_path" not in df.columns:
            raise ValueError("В CSV нет колонки crop_path")
        if "label" not in df.columns:
            raise ValueError("В CSV нет колонки label")

        df = df.copy()
        df["label"] = df["label"].map(normalize_label)
        df = df[df["label"].isin(CLASS_TO_IDX.keys())].copy()
        df["source_type"] = df.get("source_type", "csv")
        if "video" not in df.columns:
            df["video"] = None
        if "frame_idx" not in df.columns:
            df["frame_idx"] = None
        if "label_raw" not in df.columns:
            df["label_raw"] = None
        if "label_smooth" not in df.columns:
            df["label_smooth"] = None
    elif cascade_dir and cascade_dir.exists():
        df = collect_from_cascade_runs(cascade_dir)
    else:
        df = collect_from_dir(data_dir)

    if len(df) == 0:
        raise ValueError("После фильтрации не осталось ни одного ROI")

    df = df.copy()
    df["crop_path"] = df["crop_path"].astype(str)
    df = df[df["label"].isin(CLASS_TO_IDX.keys())].copy()
    df = df.drop_duplicates(subset=["crop_path"]).reset_index(drop=True)
    df["target"] = df["label"].map(CLASS_TO_IDX).astype(int)

    if cfg.MAX_SAMPLES is not None:
        df = df.sample(min(cfg.MAX_SAMPLES, len(df)), random_state=cfg.SEED).reset_index(drop=True)

    return df

roi_df = load_roi_df()

print("Число ROI:", len(roi_df))
display(roi_df.head())
display(roi_df["label"].value_counts().rename("count").to_frame())
display(roi_df["source_type"].value_counts().rename("count").to_frame())


In [ ]:
def show_random_rois(df, n=12, seed=42):
    if len(df) == 0:
        print("Нет изображений для показа")
        return

    rng = np.random.default_rng(seed)
    take = min(n, len(df))
    idx = rng.choice(len(df), size=take, replace=False)
    sample = df.iloc[idx].reset_index(drop=True)

    cols = 4
    rows = math.ceil(take / cols)
    plt.figure(figsize=(4 * cols, 3.2 * rows))

    for i, row in enumerate(sample.itertuples(index=False), 1):
        img = Image.open(row.crop_path).convert("RGB")
        plt.subplot(rows, cols, i)
        plt.imshow(img)
        plt.title(f"true={row.label}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_random_rois(roi_df, n=12, seed=cfg.SEED)

## 3. Dataset, DataLoader и базовая оценка на чистых данных

In [ ]:
class ROIDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["crop_path"]).convert("RGB")
        x = self.transform(img)
        y = int(row["target"])
        return x, y, row["crop_path"], row["label"]

def make_loader(df, shuffle=False):
    ds = ROIDataset(df, val_tf)
    return DataLoader(
        ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=shuffle,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

@torch.no_grad()
def evaluate_clean(model, df):
    loader = make_loader(df, shuffle=False)
    rows = []

    for x, y, paths, labels in loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        preds = np.argmax(probs, axis=1)

        y_np = y.numpy()
        for i in range(len(preds)):
            pred_idx = int(preds[i])
            target_idx = int(y_np[i])
            rows.append({
                "crop_path": paths[i],
                "label": labels[i],
                "target": target_idx,
                "pred": pred_idx,
                "pred_label": IDX_TO_CLASS[pred_idx],
                "correct": int(pred_idx == target_idx),
                "p_red": float(probs[i][CLASS_TO_IDX["red"]]) if "red" in CLASS_TO_IDX else None,
                "p_green": float(probs[i][CLASS_TO_IDX["green"]]) if "green" in CLASS_TO_IDX else None,
            })

    pred_df = pd.DataFrame(rows)
    acc = float(pred_df["correct"].mean()) if len(pred_df) else 0.0
    return pred_df, acc

baseline_df, baseline_acc = evaluate_clean(model, roi_df)

baseline_path = OUT_DIR / "baseline_predictions.csv"
baseline_df.to_csv(baseline_path, index=False, encoding="utf-8-sig")

print("Baseline accuracy на всём наборе:", round(baseline_acc, 4))
display(pd.crosstab(baseline_df["label"], baseline_df["pred_label"], margins=True))
print("Файл сохранён:", baseline_path)

In [ ]:
if cfg.USE_ONLY_BASELINE_CORRECT:
    controlled_df = baseline_df[baseline_df["correct"] == 1].copy().reset_index(drop=True)
else:
    controlled_df = baseline_df.copy().reset_index(drop=True)

controlled_df["target"] = controlled_df["target"].astype(int)

controlled_path = OUT_DIR / "controlled_subset.csv"
controlled_df.to_csv(controlled_path, index=False, encoding="utf-8-sig")

print("Контролируемая подвыборка:", len(controlled_df))
print("Доля от исходного набора:", round(len(controlled_df) / max(1, len(baseline_df)), 4))
display(controlled_df["label"].value_counts().rename("count").to_frame())
print("Файл сохранён:", controlled_path)

## 4. Вспомогательные функции для FGSM, BIM и PGD

In [7]:
mean_t = torch.tensor(MEAN, dtype=torch.float32, device=device).view(1, 3, 1, 1)
std_t = torch.tensor(STD, dtype=torch.float32, device=device).view(1, 3, 1, 1)
lower_bound = (0.0 - mean_t) / std_t
upper_bound = (1.0 - mean_t) / std_t

def eps_to_tensor(eps):
    return torch.tensor([eps / s for s in STD], dtype=torch.float32, device=device).view(1, 3, 1, 1)

def denormalize_batch(x):
    mean = mean_t.to(x.device)
    std = std_t.to(x.device)
    return torch.clamp(x * std + mean, 0.0, 1.0)

def clamp_normalized(x):
    return torch.max(torch.min(x, upper_bound), lower_bound)

def fgsm_attack(model, x, y, eps):
    x_orig = x.detach()
    eps_t = eps_to_tensor(eps)

    x_adv = x_orig.clone().detach().requires_grad_(True)
    logits = model(x_adv)
    loss = criterion(logits, y)

    model.zero_grad(set_to_none=True)
    loss.backward()

    grad_sign = x_adv.grad.sign()
    x_adv = x_adv + eps_t * grad_sign
    x_adv = torch.max(torch.min(x_adv, x_orig + eps_t), x_orig - eps_t)
    x_adv = clamp_normalized(x_adv).detach()
    return x_adv

def pgd_attack(model, x, y, eps, alpha, steps, random_start=False):
    x_orig = x.detach()
    eps_t = eps_to_tensor(eps)
    alpha_t = eps_to_tensor(alpha)

    if random_start:
        noise = torch.empty_like(x_orig).uniform_(-1.0, 1.0)
        x_adv = x_orig + noise * eps_t
        x_adv = clamp_normalized(x_adv)
    else:
        x_adv = x_orig.clone().detach()

    for _ in range(steps):
        x_adv.requires_grad_(True)
        logits = model(x_adv)
        loss = criterion(logits, y)

        model.zero_grad(set_to_none=True)
        loss.backward()

        with torch.no_grad():
            x_adv = x_adv + alpha_t * x_adv.grad.sign()
            x_adv = torch.max(torch.min(x_adv, x_orig + eps_t), x_orig - eps_t)
            x_adv = clamp_normalized(x_adv)

    return x_adv.detach()

def bim_attack(model, x, y, eps, alpha, steps):
    return pgd_attack(
        model=model,
        x=x,
        y=y,
        eps=eps,
        alpha=alpha,
        steps=steps,
        random_start=False,
    )

## 5. Оценка после атаки

In [8]:
def evaluate_under_attack(model, df, method, eps, alpha=None, steps=None, random_start=False):
    loader = make_loader(df, shuffle=False)
    rows = []

    for x, y, paths, labels in loader:
        x = x.to(device)
        y = y.to(device)

        if method == "fgsm":
            x_adv = fgsm_attack(model, x, y, eps)
        elif method == "bim":
            if alpha is None or steps is None:
                raise ValueError("Для BIM нужно указать alpha и steps")
            x_adv = bim_attack(model, x, y, eps, alpha=alpha, steps=steps)
        elif method == "pgd":
            if alpha is None or steps is None:
                raise ValueError("Для PGD нужно указать alpha и steps")
            x_adv = pgd_attack(model, x, y, eps, alpha=alpha, steps=steps, random_start=random_start)
        else:
            raise ValueError(f"Неизвестный метод атаки: {method}")

        with torch.no_grad():
            logits_adv = model(x_adv)
            probs_adv = torch.softmax(logits_adv, dim=1).detach().cpu().numpy()
            preds_adv = np.argmax(probs_adv, axis=1)

        y_np = y.detach().cpu().numpy()
        for i in range(len(preds_adv)):
            pred_idx = int(preds_adv[i])
            target_idx = int(y_np[i])
            rows.append({
                "method": method,
                "epsilon": float(eps),
                "alpha": float(alpha) if alpha is not None else None,
                "steps": int(steps) if steps is not None else None,
                "crop_path": paths[i],
                "label": labels[i],
                "target": target_idx,
                "pred": pred_idx,
                "pred_label": IDX_TO_CLASS[pred_idx],
                "correct": int(pred_idx == target_idx),
                "attack_success": int(pred_idx != target_idx),
                "p_red": float(probs_adv[i][CLASS_TO_IDX["red"]]) if "red" in CLASS_TO_IDX else None,
                "p_green": float(probs_adv[i][CLASS_TO_IDX["green"]]) if "green" in CLASS_TO_IDX else None,
            })

    pred_df = pd.DataFrame(rows)
    acc = float(pred_df["correct"].mean()) if len(pred_df) else 0.0
    success_rate = float(pred_df["attack_success"].mean()) if len(pred_df) else 0.0
    return pred_df, acc, success_rate

In [ ]:
results = []
attack_tables = {}

base_controlled_acc = 1.0 if cfg.USE_ONLY_BASELINE_CORRECT else float((controlled_df["pred"] == controlled_df["target"]).mean())

for eps in cfg.EPSILONS:
    # FGSM
    fgsm_df, fgsm_acc, fgsm_success = evaluate_under_attack(
        model=model,
        df=controlled_df,
        method="fgsm",
        eps=eps,
    )
    fgsm_name = f"fgsm_eps_{int(round(eps * 255))}"
    fgsm_df.to_csv(OUT_DIR / f"{fgsm_name}.csv", index=False, encoding="utf-8-sig")
    attack_tables[fgsm_name] = fgsm_df

    results.append({
        "method": "FGSM",
        "epsilon": eps,
        "alpha": None,
        "steps": None,
        "n_samples": len(fgsm_df),
        "accuracy": fgsm_acc,
        "attack_success_rate": fgsm_success,
        "accuracy_drop_abs": base_controlled_acc - fgsm_acc,
        "accuracy_drop_pct": (base_controlled_acc - fgsm_acc) * 100.0,
    })

    # BIM
    bim_df, bim_acc, bim_success = evaluate_under_attack(
        model=model,
        df=controlled_df,
        method="bim",
        eps=eps,
        alpha=cfg.BIM_ALPHA,
        steps=cfg.BIM_STEPS,
    )
    bim_name = f"bim_eps_{int(round(eps * 255))}"
    bim_df.to_csv(OUT_DIR / f"{bim_name}.csv", index=False, encoding="utf-8-sig")
    attack_tables[bim_name] = bim_df

    results.append({
        "method": "BIM",
        "epsilon": eps,
        "alpha": cfg.BIM_ALPHA,
        "steps": cfg.BIM_STEPS,
        "n_samples": len(bim_df),
        "accuracy": bim_acc,
        "attack_success_rate": bim_success,
        "accuracy_drop_abs": base_controlled_acc - bim_acc,
        "accuracy_drop_pct": (base_controlled_acc - bim_acc) * 100.0,
    })

    # PGD
    pgd_df, pgd_acc, pgd_success = evaluate_under_attack(
        model=model,
        df=controlled_df,
        method="pgd",
        eps=eps,
        alpha=cfg.PGD_ALPHA,
        steps=cfg.PGD_STEPS,
        random_start=cfg.PGD_RANDOM_START,
    )
    pgd_name = f"pgd_eps_{int(round(eps * 255))}"
    pgd_df.to_csv(OUT_DIR / f"{pgd_name}.csv", index=False, encoding="utf-8-sig")
    attack_tables[pgd_name] = pgd_df

    results.append({
        "method": "PGD",
        "epsilon": eps,
        "alpha": cfg.PGD_ALPHA,
        "steps": cfg.PGD_STEPS,
        "n_samples": len(pgd_df),
        "accuracy": pgd_acc,
        "attack_success_rate": pgd_success,
        "accuracy_drop_abs": base_controlled_acc - pgd_acc,
        "accuracy_drop_pct": (base_controlled_acc - pgd_acc) * 100.0,
    })

results_df = pd.DataFrame(results).sort_values(["method", "epsilon"]).reset_index(drop=True)
results_path = OUT_DIR / "attack_summary.csv"
results_df.to_csv(results_path, index=False, encoding="utf-8-sig")

print("Базовая accuracy на контролируемой подвыборке:", round(base_controlled_acc, 4))
display(results_df)
print("Файл сохранён:", results_path)

In [ ]:
plt.figure(figsize=(8, 4))
for method in ["FGSM", "BIM", "PGD"]:
    part = results_df[results_df["method"] == method].sort_values("epsilon")
    plt.plot(part["epsilon"] * 255, part["accuracy"], marker="o", label=method)

plt.title("Точность классификации после атаки")
plt.xlabel("epsilon, уровни яркости пикселя")
plt.ylabel("Accuracy")
plt.xticks([2, 4, 8], ["2/255", "4/255", "8/255"])
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
for method in ["FGSM", "BIM", "PGD"]:
    part = results_df[results_df["method"] == method].sort_values("epsilon")
    plt.plot(part["epsilon"] * 255, part["attack_success_rate"], marker="o", label=method)

plt.title("Доля успешных состязательных атак")
plt.xlabel("epsilon, уровни яркости пикселя")
plt.ylabel("Attack success rate")
plt.xticks([2, 4, 8], ["2/255", "4/255", "8/255"])
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()
plt.show()

## 6. Наглядные примеры для работы

In [11]:
def tensor_to_image_np(x):
    img = denormalize_batch(x.unsqueeze(0))[0].detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(img, 0.0, 1.0)

def load_one_tensor(image_path):
    img = Image.open(image_path).convert("RGB")
    x = val_tf(img)
    return x

@torch.no_grad()
def predict_one_tensor(model, x):
    logits = model(x.unsqueeze(0).to(device))
    probs = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()
    pred_idx = int(np.argmax(probs))
    return {
        "pred_idx": pred_idx,
        "pred_label": IDX_TO_CLASS[pred_idx],
        "probs": probs,
    }

def build_adv_for_path(image_path, target_idx, method, eps, alpha=None, steps=None):
    x = load_one_tensor(image_path).unsqueeze(0).to(device)
    y = torch.tensor([target_idx], dtype=torch.long, device=device)

    if method == "fgsm":
        x_adv = fgsm_attack(model, x, y, eps=eps)
    elif method == "bim":
        if alpha is None or steps is None:
            raise ValueError("Для BIM нужно указать alpha и steps")
        x_adv = bim_attack(model, x, y, eps=eps, alpha=alpha, steps=steps)
    elif method == "pgd":
        if alpha is None or steps is None:
            raise ValueError("Для PGD нужно указать alpha и steps")
        x_adv = pgd_attack(
            model,
            x,
            y,
            eps=eps,
            alpha=alpha,
            steps=steps,
            random_start=cfg.PGD_RANDOM_START
        )
    else:
        raise ValueError(method)

    return x[0].detach().cpu(), x_adv[0].detach().cpu()

def show_attack_gallery(attack_df, method, eps, n=6, only_success=True, save_name=None):
    view = attack_df.copy()
    if only_success:
        view = view[view["attack_success"] == 1].copy()

    if len(view) == 0:
        print("Нет подходящих примеров для визуализации")
        return

    take = min(n, len(view))
    sample = view.sample(take, random_state=cfg.SEED).reset_index(drop=True)

    rows = take
    cols = 3
    plt.figure(figsize=(12, 4 * rows))

    for i, row in enumerate(sample.itertuples(index=False), 1):
        if method == "bim":
            alpha = cfg.BIM_ALPHA
            steps = cfg.BIM_STEPS
        elif method == "pgd":
            alpha = cfg.PGD_ALPHA
            steps = cfg.PGD_STEPS
        else:
            alpha = None
            steps = None

        x_clean, x_adv = build_adv_for_path(
            image_path=row.crop_path,
            target_idx=int(row.target),
            method=method,
            eps=eps,
            alpha=alpha,
            steps=steps,
        )

        pred_clean = predict_one_tensor(model, x_clean)
        pred_adv = predict_one_tensor(model, x_adv)

        img_clean = tensor_to_image_np(x_clean)
        img_adv = tensor_to_image_np(x_adv)
        delta = np.abs(img_adv - img_clean)
        if delta.max() > 0:
            delta = delta / (delta.max() + 1e-8)

        plt.subplot(rows, cols, (i - 1) * cols + 1)
        plt.imshow(img_clean)
        plt.title(f"Оригинал\ntrue={row.label}\npred={pred_clean['pred_label']}")
        plt.axis("off")

        plt.subplot(rows, cols, (i - 1) * cols + 2)
        plt.imshow(img_adv)
        plt.title(
            f"{method.upper()} eps={eps*255:.0f}/255\n"
            f"pred={pred_adv['pred_label']}"
        )
        plt.axis("off")

        plt.subplot(rows, cols, (i - 1) * cols + 3)
        plt.imshow(delta)
        plt.title("Усиленная разность")
        plt.axis("off")

    plt.tight_layout()

    if save_name is not None:
        save_path = OUT_DIR / save_name
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Иллюстрация сохранена:", save_path)

    plt.show()

In [ ]:
# Примеры визуализации:
eps_last = max(cfg.EPSILONS)

fgsm_key = f"fgsm_eps_{int(round(eps_last * 255))}"
bim_key  = f"bim_eps_{int(round(eps_last * 255))}"
pgd_key  = f"pgd_eps_{int(round(eps_last * 255))}"

show_attack_gallery(
    attack_df=attack_tables[fgsm_key],
    method="fgsm",
    eps=eps_last,
    n=cfg.VIS_SAMPLES_PER_SETTING,
    only_success=True,
    save_name=f"gallery_{fgsm_key}.png",
)

show_attack_gallery(
    attack_df=attack_tables[bim_key],
    method="bim",
    eps=eps_last,
    n=cfg.VIS_SAMPLES_PER_SETTING,
    only_success=True,
    save_name=f"gallery_{bim_key}.png",
)

show_attack_gallery(
    attack_df=attack_tables[pgd_key],
    method="pgd",
    eps=eps_last,
    n=cfg.VIS_SAMPLES_PER_SETTING,
    only_success=True,
    save_name=f"gallery_{pgd_key}.png",
)

## 7. Готовая компактная таблица

In [ ]:
report_df = results_df.copy()
report_df["epsilon_255"] = (report_df["epsilon"] * 255).round().astype(int).astype(str) + "/255"
report_df["accuracy"] = report_df["accuracy"].round(4)
report_df["attack_success_rate"] = report_df["attack_success_rate"].round(4)
report_df["accuracy_drop_abs"] = report_df["accuracy_drop_abs"].round(4)
report_df["accuracy_drop_pct"] = report_df["accuracy_drop_pct"].round(2)

report_df = report_df[
    ["method", "epsilon_255", "n_samples", "accuracy", "attack_success_rate", "accuracy_drop_abs", "accuracy_drop_pct"]
].rename(columns={
    "method": "Метод",
    "epsilon_255": "epsilon",
    "n_samples": "Число ROI",
    "accuracy": "Accuracy",
    "attack_success_rate": "Доля успешных атак",
    "accuracy_drop_abs": "Падение accuracy (абс.)",
    "accuracy_drop_pct": "Падение accuracy, %",
})

report_path = OUT_DIR / "attack_report_table.csv"
report_df.to_csv(report_path, index=False, encoding="utf-8-sig")

display(report_df)
print("Файл сохранён:", report_path)

## 8. Краткие выводы по эксперименту

In [ ]:
if len(results_df):
    best_fgsm = results_df[results_df["method"] == "FGSM"].sort_values("epsilon").iloc[-1]
    best_pgd = results_df[results_df["method"] == "PGD"].sort_values("epsilon").iloc[-1]

    print("Черновик выводов:")
    print()
    print(
        f"1. На контролируемой подвыборке корректно локализованных ROI базовая точность "
        f"классификатора составила {base_controlled_acc:.4f}."
    )
    print(
        f"2. Одношаговая атака FGSM при epsilon={int(round(best_fgsm['epsilon'] * 255))}/255 "
        f"снизила accuracy до {best_fgsm['accuracy']:.4f}, "
        f"что соответствует падению на {best_fgsm['accuracy_drop_pct']:.2f}%."
    )
    print(
        f"3. Итерационная атака PGD при epsilon={int(round(best_pgd['epsilon'] * 255))}/255 "
        f"снизила accuracy до {best_pgd['accuracy']:.4f}, "
        f"что соответствует падению на {best_pgd['accuracy_drop_pct']:.2f}%."
    )
    print(
        "4. Полученные результаты подтверждают уязвимость классификационного модуля "
        "ResNet-18 к малым состязательным возмущениям входных изображений."
    )
else:
    print("Нет результатов для формирования выводов.")